# 🤖 FinGPT Fine-Tuning Notebook
## LoRA Supervised Fine-Tuning + RLHF (PPO) Alignment

**Base Model**: `Qwen/Qwen1.5-0.5B` (Apache 2.0, ~1GB, no gating required)  
**Dataset**: `FinGPT/fingpt-sentiment-train` (financial sentiment instruction pairs)  
**Techniques**: QLoRA (4-bit quantization) → SFT → PPO-based RLHF alignment  
**Output**: Saved LoRA adapter to `./backend/fingpt-lora/`

---
> ⚠️ **Hardware note**: This notebook is designed to run on CPU or a single GPU (≥6GB VRAM).  
> 4-bit quantization via `bitsandbytes` is used to minimize memory usage.  
> If running on CPU only, training will be slow but will complete.

In [ ]:
# ============================================================
# CELL 1: Install Required Dependencies
# ============================================================
# We install all necessary packages for:
#   - transformers: HuggingFace model loading & tokenization
#   - peft:         Parameter-Efficient Fine-Tuning (LoRA)
#   - trl:          Transformer Reinforcement Learning (SFT + PPO)
#   - bitsandbytes: 4-bit/8-bit quantization (QLoRA)
#   - datasets:     HuggingFace dataset hub
#   - accelerate:   Distributed/mixed-precision training
#   - rouge-score:  ROUGE-L evaluation metric
#   - bert-score:   BERTScore semantic similarity metric

print("📦 Installing dependencies...")

import subprocess, sys

packages = [
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "trl>=0.8.6",
    "bitsandbytes>=0.43.0",
    "datasets>=2.18.0",
    "accelerate>=0.29.0",
    "rouge-score",
    "bert-score",
    "scipy",
    "sentencepiece",
    "einops",
]

for pkg in packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        check=False
    )

print("✅ All packages installed successfully!")

: 

In [ ]:
# ============================================================
# CELL 2: Configuration & Hyperparameters
# ============================================================
# This cell defines ALL configurable parameters in one place.
# Changing values here will propagate throughout the notebook.

import os
import warnings
warnings.filterwarnings("ignore")

# ── Model ────────────────────────────────────────────────────
# Qwen1.5-0.5B: Apache 2.0 license, no HuggingFace gating,
# only ~1GB — ideal for CPU/low-VRAM environments.
BASE_MODEL_NAME  = "Qwen/Qwen1.5-0.5B"

# ── Output paths ────────────────────────────────────────────
SFT_OUTPUT_DIR   = "./backend/fingpt-lora"          # SFT adapter path (used by backend)
PPO_OUTPUT_DIR   = "./backend/fingpt-lora-rlhf"     # PPO-aligned adapter path
FINAL_OUTPUT_DIR = SFT_OUTPUT_DIR                   # Backend will load from here

# ── Dataset ─────────────────────────────────────────────────
DATASET_NAME     = "FinGPT/fingpt-sentiment-train"  # Official FinGPT sentiment dataset
MAX_SAMPLES      = 2000   # Limit samples for faster training (set None for full dataset)
TEST_SPLIT_SIZE  = 0.1    # 10% held out for evaluation
MAX_SEQ_LENGTH   = 256    # Max tokens per training example

# ── LoRA hyperparameters ─────────────────────────────────────
LORA_R           = 16     # LoRA rank (higher = more expressive, more memory)
LORA_ALPHA       = 32     # LoRA scaling factor (alpha/r = 2 is standard)
LORA_DROPOUT     = 0.05   # Dropout for regularization
LORA_TARGET      = ["q_proj", "v_proj"]  # Which attention layers to adapt

# ── SFT training args ────────────────────────────────────────
SFT_EPOCHS       = 1      # Epochs (1 is enough for demo; use 3+ for production)
SFT_BATCH_SIZE   = 4      # Per-device batch size
SFT_LR           = 2e-4   # Learning rate (higher works well with LoRA)
SFT_GRAD_ACCUM   = 4      # Gradient accumulation steps (effective batch = 16)
SFT_FP16         = False  # Set True if CUDA GPU available with 16-bit support

# ── PPO (RLHF) args ──────────────────────────────────────────
PPO_STEPS        = 20     # PPO update steps (short loop for demo)
PPO_BATCH_SIZE   = 4      # Batch size per PPO step
PPO_LR           = 1.4e-5 # PPO learning rate (lower than SFT)

# ── Reward model ─────────────────────────────────────────────
# Using 'distilbert-base-uncased-finetuned-sst-2-english' as a
# proxy reward model (positive sentiment = good financial tone)
REWARD_MODEL     = "distilbert-base-uncased-finetuned-sst-2-english"

# ── HuggingFace token (optional for some datasets) ───────────
HF_TOKEN = os.getenv("HF_TOKEN", "")  # Set via env if needed

print("📋 Configuration loaded:")
print(f"   Base model     : {BASE_MODEL_NAME}")
print(f"   Dataset        : {DATASET_NAME}")
print(f"   Max samples    : {MAX_SAMPLES}")
print(f"   LoRA rank      : {LORA_R}, alpha: {LORA_ALPHA}")
print(f"   SFT epochs     : {SFT_EPOCHS}, lr: {SFT_LR}")
print(f"   PPO steps      : {PPO_STEPS}")
print(f"   Output dir     : {SFT_OUTPUT_DIR}")
print("✅ Config ready!")

In [ ]:
# ============================================================
# CELL 3: Load Tokenizer
# ============================================================
# The tokenizer converts raw text into token IDs that the model
# understands. We also set special tokens needed for padding.

from transformers import AutoTokenizer

print(f"⏳ Loading tokenizer from '{BASE_MODEL_NAME}'...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,   # Required for Qwen's custom code
    use_fast=True,            # Use the Rust-based fast tokenizer
)

# Qwen models often don't have a default pad token.
# We use the EOS token as the padding token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print("   ℹ️ Set pad_token = eos_token")

# Use right-side padding (required for causal LMs like GPT-style models)
tokenizer.padding_side = "right"

print(f"✅ Tokenizer loaded!")
print(f"   Vocabulary size : {tokenizer.vocab_size:,}")
print(f"   Pad token       : '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"   EOS token       : '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")

# Quick sanity check: encode a sample financial sentence
sample = "The stock market rallied strongly today on positive earnings reports."
encoded = tokenizer(sample, return_tensors="pt")
print(f"\n   Sample encode test: '{sample[:50]}...'")
print(f"   → {encoded['input_ids'].shape[1]} tokens")

In [ ]:
# ============================================================
# CELL 4: Load Base Model with QLoRA (4-bit Quantization)
# ============================================================
# QLoRA combines:
#   - 4-bit weight quantization (reduces memory 4x vs float16)
#   - LoRA adapters (trains only small adapter matrices)
# This allows fine-tuning a 0.5B model on even a laptop CPU.

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Detect device
device_str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Running on device: {device_str.upper()}")
if device_str == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️ No GPU found — running on CPU (slower but works)")

# ─── QLoRA quantization config (only effective on GPU) ────────
# On CPU, bitsandbytes quantization isn't supported, so we load
# in full precision (float32) automatically.
if device_str == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,               # Load weights in 4-bit
        bnb_4bit_quant_type="nf4",       # NormalFloat4 quant (best quality)
        bnb_4bit_compute_dtype=torch.float16,  # Compute in fp16
        bnb_4bit_use_double_quant=True,  # Nested quantization for even less memory
    )
    load_kwargs = {"quantization_config": bnb_config, "device_map": "auto"}
    print("   ✓ Using 4-bit QLoRA quantization")
else:
    bnb_config = None
    load_kwargs = {"torch_dtype": torch.float32}
    print("   ✓ Loading in float32 (CPU mode)")

print(f"\n⏳ Downloading & loading '{BASE_MODEL_NAME}'...")
print("   (This will download ~1GB on first run — please wait)")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,
    **load_kwargs,
)

# Disable model caching (not needed during training)
model.config.use_cache = False
model.config.pretraining_tp = 1  # Tensor parallel = 1 (single GPU)

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Base model loaded!")
print(f"   Model type       : {model.__class__.__name__}")
print(f"   Total parameters : {total_params:,}")
print(f"   Trainable params : {trainable_params:,}")
print(f"   Architecture     : {model.config.architectures}")

In [ ]:
# ============================================================
# CELL 5: Prepare & Format the FinGPT Dataset
# ============================================================
# We use the official FinGPT sentiment training dataset from HuggingFace.
# Each sample is formatted as an instruction-response pair, which is the
# standard format for fine-tuning chat/instruction-following LLMs.
#
# Format:
#   ### Instruction:\n{instruction}\n\n### Response:\n{output}

from datasets import load_dataset
import random

print(f"⏳ Loading dataset '{DATASET_NAME}'...")

# Load the dataset (train split only)
raw_dataset = load_dataset(DATASET_NAME, split="train", trust_remote_code=True)

print(f"   Raw dataset size: {len(raw_dataset):,} samples")
print(f"   Columns: {raw_dataset.column_names}")

# Limit dataset size for manageable training
if MAX_SAMPLES and len(raw_dataset) > MAX_SAMPLES:
    raw_dataset = raw_dataset.shuffle(seed=42).select(range(MAX_SAMPLES))
    print(f"   ✂️ Reduced to {MAX_SAMPLES} samples (for faster training)")

# ─── Format function: creates instruction-response text ───────
def format_instruction(sample):
    """
    Converts raw FinGPT sample into a structured instruction-following prompt.
    The model learns to generate financial sentiment analysis as an assistant.
    
    FinGPT dataset fields:
      'input'  : the financial text/news headline
      'output' : the sentiment label (positive/negative/neutral)
    """
    instruction = sample.get("instruction", "What is the sentiment of this financial text?")
    context     = sample.get("input", "")
    response    = sample.get("output", "neutral")
    
    # Build the full training text in Alpaca-style format
    text = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{context}\n\n"
        f"### Response:\n{response}"
        f"{tokenizer.eos_token}"  # End-of-sequence marker
    )
    return {"text": text}

# Apply formatting to all samples
print("\n⏳ Formatting dataset as instruction-response pairs...")
formatted_dataset = raw_dataset.map(
    format_instruction,
    remove_columns=raw_dataset.column_names,  # Drop old columns, keep only 'text'
    desc="Formatting",
)

# Split into train / test sets
split = formatted_dataset.train_test_split(
    test_size=TEST_SPLIT_SIZE,
    seed=42
)
train_dataset = split["train"]
test_dataset  = split["test"]

print(f"\n✅ Dataset ready!")
print(f"   Train samples : {len(train_dataset):,}")
print(f"   Test samples  : {len(test_dataset):,}")
print(f"\n📋 Sample formatted training example:")
print("-" * 60)
print(train_dataset[0]["text"][:400])
print("-" * 60)

In [ ]:
# ============================================================
# CELL 6: Apply LoRA Adapters (Parameter-Efficient Fine-Tuning)
# ============================================================
# LoRA (Low-Rank Adaptation) works by injecting small trainable
# matrices (rank-r decomposition) into the attention layers.
#
# Instead of updating ALL ~500M parameters, we only update
# the LoRA adapters (~0.5M parameters = ~0.1% of total).
# This is ~1000x more efficient than full fine-tuning!
#
# LoRA math: W' = W + α/r * (A × B)
#   W  = frozen pretrained weight
#   A  = trainable low-rank matrix (r × d)
#   B  = trainable low-rank matrix (d × r)
#   α  = scaling factor

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print("🔧 Applying LoRA adapters to the model...")

# Prepare model for k-bit training (only needed if quantized)
if device_str == "cuda" and bnb_config is not None:
    model = prepare_model_for_kbit_training(model)
    print("   ✓ Model prepared for k-bit (QLoRA) training")

# ─── LoRA configuration ───────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # We're doing causal language modeling
    r=LORA_R,                       # Rank of LoRA decomposition
    lora_alpha=LORA_ALPHA,          # Scaling factor
    target_modules=LORA_TARGET,     # Which projection matrices to adapt
    lora_dropout=LORA_DROPOUT,      # Dropout for regularization
    bias="none",                    # Don't train bias parameters
    inference_mode=False,           # We are in training mode
)

# Wrap the base model with LoRA
model = get_peft_model(model, lora_config)

# Count how many parameters are actually trainable now
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params       = sum(p.numel() for p in model.parameters())
pct_trainable    = 100 * trainable_params / all_params

print(f"\n✅ LoRA adapters applied!")
print(f"   Total parameters     : {all_params:,}")
print(f"   Trainable (LoRA)     : {trainable_params:,}")
print(f"   % Trainable          : {pct_trainable:.4f}%")
print(f"   LoRA rank            : {LORA_R}")
print(f"   LoRA alpha           : {LORA_ALPHA}")
print(f"   Target modules       : {LORA_TARGET}")
print(f"\n   💡 Only {pct_trainable:.4f}% of parameters will be updated — this is the power of LoRA!")

In [ ]:
# ============================================================
# CELL 7: Supervised Fine-Tuning (SFT) with SFTTrainer
# ============================================================
# SFT is the first phase: we teach the model to follow financial
# instructions using supervised examples from the dataset.
#
# SFTTrainer from TRL library handles:
#   - Automatic data collation and tokenization
#   - Gradient accumulation
#   - Loss logging
#   - Checkpoint saving

from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

print("🏋️ Starting Supervised Fine-Tuning (SFT) with LoRA...")
print(f"   Training on {len(train_dataset):,} samples for {SFT_EPOCHS} epoch(s)")
print(f"   This may take several minutes...\n")

# ─── Training configuration ───────────────────────────────────
sft_config = SFTConfig(
    # Output & saving
    output_dir=SFT_OUTPUT_DIR,
    
    # Training duration
    num_train_epochs=SFT_EPOCHS,
    max_steps=-1,                    # -1 = train for full epochs
    
    # Batch size & gradient accumulation
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,  # Effective batch = 4*4 = 16
    
    # Optimizer
    learning_rate=SFT_LR,
    lr_scheduler_type="cosine",      # Cosine decay for smooth LR schedule
    warmup_ratio=0.05,               # 5% of steps for linear warmup
    weight_decay=0.01,               # L2 regularization
    optim="adamw_torch",             # AdamW optimizer (best for LLMs)
    
    # Precision
    fp16=SFT_FP16,
    bf16=False,                      # Use bfloat16 if on Ampere GPU
    
    # Sequence
    max_seq_length=MAX_SEQ_LENGTH,
    
    # Logging
    logging_steps=10,                # Log loss every 10 steps
    logging_dir="./logs/sft",
    report_to="none",                # Disable wandb/tensorboard
    
    # Saving
    save_strategy="epoch",
    save_total_limit=1,              # Keep only the latest checkpoint
    
    # Misc
    seed=42,
    group_by_length=True,            # Group similar-length samples (faster)
    dataset_text_field="text",       # Our formatted text column
)

# ─── Initialize the SFT trainer ──────────────────────────────
sft_trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    args=sft_config,
)

# ─── Train! ──────────────────────────────────────────────────
print("🚀 Training started...")
sft_result = sft_trainer.train()

print(f"\n✅ SFT Training complete!")
print(f"   Final training loss : {sft_result.training_loss:.4f}")
print(f"   Total steps         : {sft_result.global_step}")
print(f"   Runtime             : {sft_result.metrics.get('train_runtime', 'N/A'):.1f}s")
print(f"   Samples/second      : {sft_result.metrics.get('train_samples_per_second', 0):.2f}")

In [ ]:
# ============================================================
# CELL 8: Save the SFT LoRA Adapter
# ============================================================
# We save ONLY the LoRA adapter weights (not the full model).
# This is tiny (~15MB) compared to the full model (~1GB).
#
# At inference time, we:
#   1. Load the base model (Qwen1.5-0.5B)
#   2. Load this adapter on top
#   3. Optionally merge them with `.merge_and_unload()`

import os

print(f"💾 Saving SFT LoRA adapter to '{SFT_OUTPUT_DIR}'...")

# Save the PEFT model (adapter only)
sft_trainer.model.save_pretrained(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)

# Save training config for reproducibility
import json
meta = {
    "base_model": BASE_MODEL_NAME,
    "training_type": "SFT + LoRA",
    "dataset": DATASET_NAME,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "max_seq_length": MAX_SEQ_LENGTH,
    "epochs": SFT_EPOCHS,
    "final_loss": round(sft_result.training_loss, 4),
}
with open(os.path.join(SFT_OUTPUT_DIR, "fingpt_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

# List saved files
saved_files = os.listdir(SFT_OUTPUT_DIR)
print(f"\n✅ Adapter saved successfully!")
print(f"   Location : {os.path.abspath(SFT_OUTPUT_DIR)}")
print(f"   Files    : {saved_files}")

# Compute total size
total_size = sum(
    os.path.getsize(os.path.join(SFT_OUTPUT_DIR, f))
    for f in saved_files
    if os.path.isfile(os.path.join(SFT_OUTPUT_DIR, f))
)
print(f"   Total size: {total_size / 1e6:.1f} MB")
print(f"\n   🎉 This adapter will be loaded by the MarketLens backend!")

In [ ]:
# ============================================================
# CELL 9: RLHF Phase — Reward Model Setup
# ============================================================
# RLHF (Reinforcement Learning from Human Feedback) is the second
# alignment phase. After SFT, the model knows HOW to answer,
# but RLHF teaches it to answer in a way humans PREFER.
#
# Standard RLHF pipeline:
#   1. ✅ SFT: Supervised fine-tuning (done above)
#   2. 🏆 Reward Model: Trained on human preference pairs (A vs B)
#   3. 🔄 PPO: The LLM is optimized to maximize reward
#
# For this notebook we use a PROXY reward model:
#   'distilbert-base-uncased-finetuned-sst-2-english'
#   → A DistilBERT fine-tuned for positive/negative sentiment
#   → Higher positive sentiment score = higher reward
#   → This simulates preferring confident, positive-toned financial responses
#
# In production: replace with a model trained on real human preference data.

from transformers import pipeline as hf_pipeline

print("🏆 Loading proxy reward model for RLHF...")
print(f"   Reward model: {REWARD_MODEL}")
print("   (Using SST-2 sentiment as a proxy — positive tone = higher reward)")

# Load reward model as a sentiment classifier
reward_pipe = hf_pipeline(
    "sentiment-analysis",
    model=REWARD_MODEL,
    device=-1,             # Always CPU for reward model (saves GPU for LLM)
    truncation=True,
    max_length=256,
)

def compute_reward(texts: list) -> list:
    """
    Compute a scalar reward for each generated text.
    
    Reward logic:
      - POSITIVE sentiment → reward = +score (encourages confident tone)
      - NEGATIVE sentiment → reward = -score (penalizes negative tone)
    
    Rewards are clamped to [-1, +1] to prevent reward hacking.
    """
    results = reward_pipe(texts)
    rewards = []
    for r in results:
        label = r["label"]   # "POSITIVE" or "NEGATIVE"
        score = r["score"]   # Confidence [0, 1]
        reward = score if label == "POSITIVE" else -score
        rewards.append(reward)
    return rewards

# Test the reward model
test_texts = [
    "NVDA shows strong bullish momentum with excellent earnings.",
    "The market crashed causing significant investor losses.",
    "The price remained relatively flat with neutral performance.",
]
test_rewards = compute_reward(test_texts)

print(f"\n✅ Reward model loaded!")
print(f"\n   Reward test examples:")
for text, reward in zip(test_texts, test_rewards):
    bar = "█" * int(abs(reward) * 10)
    sign = "+" if reward > 0 else "-"
    print(f"   [{sign}{abs(reward):.3f}] {bar} | {text[:55]}")

In [ ]:
# ============================================================
# CELL 10: PPO Training (RLHF Alignment Loop)
# ============================================================
# PPO (Proximal Policy Optimization) is the RL algorithm used to
# align the LLM with the reward signal from the reward model.
#
# Each PPO step:
#   1. Sample a query from the dataset
#   2. Generate a response with the current LLM
#   3. Score it with the reward model
#   4. Update the LLM to maximize reward (with KL penalty)
#
# The KL divergence penalty prevents the model from drifting
# too far from the SFT baseline (reward hacking prevention).

from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
import torch

print("🔄 Setting up PPO trainer for RLHF alignment...")

# Load the SFT model with a value head (needed for PPO)
# The value head predicts expected future reward (critic network)
print("   Loading SFT model with value head...")
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    SFT_OUTPUT_DIR,              # Load from our saved SFT adapter
    trust_remote_code=True,
    torch_dtype=torch.float32,   # Full precision for stability
    is_trainable=True,
)

# ─── PPO configuration ───────────────────────────────────────
ppo_config = PPOConfig(
    model_name=SFT_OUTPUT_DIR,
    learning_rate=PPO_LR,
    batch_size=PPO_BATCH_SIZE,
    mini_batch_size=1,             # Process 1 sample at a time per mini-batch
    gradient_accumulation_steps=1,
    ppo_epochs=1,                  # PPO clip epochs per update
    kl_penalty="kl",               # KL divergence penalty (prevents reward hacking)
    init_kl_coef=0.2,              # Initial KL coefficient
    target_kl=6.0,                 # Target KL (adaptive controller)
    seed=42,
)

# Initialize PPO trainer
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=None,                # ref_model=None → uses frozen copy of initial model
    tokenizer=tokenizer,
    dataset=train_dataset,
    data_collator=lambda data: {"input_ids": [d["input_ids"] for d in data]},
)

# ─── PPO training loop ───────────────────────────────────────
print(f"\n🚀 Starting PPO alignment loop ({PPO_STEPS} steps)...")
print("   Each step: sample query → generate response → compute reward → update")

ppo_rewards_log = []
ppo_loss_log    = []

gen_kwargs = {
    "max_new_tokens": 64,
    "do_sample": True,
    "temperature": 0.7,
    "pad_token_id": tokenizer.pad_token_id,
}

for step, batch in enumerate(ppo_trainer.dataloader):
    if step >= PPO_STEPS:
        break
    
    # 1. Tokenize input queries
    query_tensors = [
        tokenizer(
            text[:MAX_SEQ_LENGTH], return_tensors="pt", truncation=True
        ).input_ids.squeeze(0)
        for text in batch["text"]
    ]
    
    # 2. Generate responses from the current policy (LLM)
    response_tensors = ppo_trainer.generate(query_tensors, **gen_kwargs)
    
    # 3. Decode generated text
    batch_responses = [
        tokenizer.decode(r.squeeze(), skip_special_tokens=True)
        for r in response_tensors
    ]
    
    # 4. Compute rewards from reward model
    rewards_list = compute_reward(batch_responses)
    reward_tensors = [torch.tensor(r, dtype=torch.float32) for r in rewards_list]
    
    # 5. PPO update step
    stats = ppo_trainer.step(query_tensors, response_tensors, reward_tensors)
    
    # Log progress
    mean_reward = sum(rewards_list) / len(rewards_list)
    ppo_rewards_log.append(mean_reward)
    
    if (step + 1) % 5 == 0 or step == 0:
        print(f"   Step {step+1:3d}/{PPO_STEPS} | Mean reward: {mean_reward:+.4f} | " +
              f"KL: {stats.get('objective/kl', 0):.4f}")

avg_reward = sum(ppo_rewards_log) / len(ppo_rewards_log) if ppo_rewards_log else 0
print(f"\n✅ PPO training complete!")
print(f"   Total PPO steps    : {len(ppo_rewards_log)}")
print(f"   Average reward     : {avg_reward:+.4f}")
print(f"   Reward trend       : {'📈 Improving' if len(ppo_rewards_log) > 1 and ppo_rewards_log[-1] > ppo_rewards_log[0] else '📊 Stable'}")

In [ ]:
# ============================================================
# CELL 11: Save Final PPO-Aligned Model
# ============================================================
# Save the PPO-aligned model. We update the SFT adapter directory
# so the backend loads the RLHF-improved version.
#
# Updated metadata records that this went through both SFT + RLHF.

import json, os

print("💾 Saving PPO-aligned model...")

# Save the full model (with PPO value head removed)
ppo_trainer.model.pretrained_model.save_pretrained(PPO_OUTPUT_DIR)
tokenizer.save_pretrained(PPO_OUTPUT_DIR)

# Also overwrite the SFT dir so backend uses the RLHF-tuned version
ppo_trainer.model.pretrained_model.save_pretrained(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)

# Update metadata
meta_path = os.path.join(FINAL_OUTPUT_DIR, "fingpt_meta.json")
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
else:
    meta = {}

meta.update({
    "training_type": "SFT + LoRA + RLHF (PPO)",
    "rlhf_steps": len(ppo_rewards_log),
    "rlhf_avg_reward": round(avg_reward, 4),
    "reward_model": REWARD_MODEL,
    "final_output_dir": os.path.abspath(FINAL_OUTPUT_DIR),
})
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"\n✅ Final RLHF model saved!")
print(f"   SFT adapter  → {os.path.abspath(SFT_OUTPUT_DIR)}")
print(f"   RLHF adapter → {os.path.abspath(PPO_OUTPUT_DIR)}")
print(f"   Backend will load from: {os.path.abspath(FINAL_OUTPUT_DIR)}")
print(f"\n   Training pipeline: SFT (LoRA) → RLHF (PPO) ✓")

In [ ]:
# ============================================================
# CELL 12: Evaluation — ROUGE-L, BERTScore, Perplexity
# ============================================================
# Evaluate the fine-tuned model on the held-out test set.
#
# Metrics:
#   📏 ROUGE-L   : Longest Common Subsequence overlap (text similarity)
#                  Range: 0-1 (higher is better)
#   🤖 BERTScore : Semantic similarity using BERT embeddings
#                  Range: 0-1 (higher is better)
#   📉 Perplexity: How surprised the model is by the test text
#                  Range: 1-∞ (lower is better; good LLMs: 5-30)

import torch
import math
from transformers import AutoModelForCausalLM
from peft import PeftModel
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

print("📊 Running evaluation on test set...")
print(f"   Test samples: {len(test_dataset):,}")
NUM_EVAL = min(50, len(test_dataset))  # Evaluate on up to 50 samples
eval_samples = test_dataset.select(range(NUM_EVAL))

# ─── Load the fine-tuned model for generation ─────────────────
print("   Loading fine-tuned model for generation...")
eval_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, trust_remote_code=True,
    torch_dtype=torch.float32
)
eval_model = PeftModel.from_pretrained(eval_base, FINAL_OUTPUT_DIR)
eval_model = eval_model.merge_and_unload()  # Merge LoRA into base for fast inference
eval_model.eval()

# ─── Generate predictions ─────────────────────────────────────
predictions, references = [], []

print("   Generating responses...")
for i, sample in enumerate(eval_samples):
    full_text = sample["text"]
    
    # Extract the prompt (everything before "### Response:")
    if "### Response:" in full_text:
        prompt_part   = full_text.split("### Response:")[0] + "### Response:"
        reference_part = full_text.split("### Response:")[1].strip()
    else:
        prompt_part   = full_text[:200]
        reference_part = full_text[200:]
    
    # Tokenize and generate
    inputs = tokenizer(prompt_part, return_tensors="pt", truncation=True, max_length=200)
    with torch.no_grad():
        output_ids = eval_model.generate(
            inputs.input_ids,
            max_new_tokens=32,
            do_sample=False,          # Greedy decoding for reproducibility
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    predictions.append(prediction)
    references.append(reference_part[:200])  # Truncate reference for fair comparison
    
    if (i+1) % 10 == 0:
        print(f"   Generated {i+1}/{NUM_EVAL}...")

# ─── ROUGE-L ─────────────────────────────────────────────────
print("\n   Computing ROUGE-L...")
scorer_obj = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge_scores = [
    scorer_obj.score(ref, pred)["rougeL"].fmeasure
    for ref, pred in zip(references, predictions)
]
avg_rouge_l = sum(rouge_scores) / len(rouge_scores)

# ─── BERTScore ───────────────────────────────────────────────
print("   Computing BERTScore (this may take a few minutes)...")
try:
    P, R, F1 = bert_score_fn(
        predictions, references,
        lang="en",
        model_type="distilbert-base-uncased",  # Lightweight BERT model
        verbose=False,
    )
    avg_bertscore_f1 = F1.mean().item()
except Exception as e:
    print(f"   ⚠️ BERTScore failed: {e}")
    avg_bertscore_f1 = float("nan")

# ─── Perplexity ──────────────────────────────────────────────
print("   Computing Perplexity...")
total_loss = 0.0
n_tokens   = 0
for sample in eval_samples:
    enc = tokenizer(
        sample["text"], return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LENGTH
    )
    input_ids = enc.input_ids
    with torch.no_grad():
        out = eval_model(input_ids, labels=input_ids)
    total_loss += out.loss.item() * input_ids.shape[1]
    n_tokens   += input_ids.shape[1]
avg_perplexity = math.exp(total_loss / n_tokens) if n_tokens > 0 else float("inf")

print("\n✅ Evaluation complete!")

In [ ]:
# ============================================================
# CELL 13: Results Summary
# ============================================================
# Print a complete summary of the fine-tuning pipeline and
# all evaluation metrics in a clean, readable format.

print("=" * 65)
print(" FinGPT Fine-Tuning — Complete Results Summary")
print("=" * 65)

print("\n📦 Model & Training Config")
print(f"   Base model       : {BASE_MODEL_NAME}")
print(f"   Dataset          : {DATASET_NAME}")
print(f"   Train samples    : {len(train_dataset):,}")
print(f"   Test samples     : {len(test_dataset):,}")
print(f"   LoRA rank / alpha: {LORA_R} / {LORA_ALPHA}")
print(f"   SFT epochs       : {SFT_EPOCHS}, lr={SFT_LR}")
print(f"   PPO steps        : {len(ppo_rewards_log)}, lr={PPO_LR}")

print("\n🏋️ Training Results")
print(f"   SFT final loss   : {sft_result.training_loss:.4f}")
print(f"   RLHF avg reward  : {avg_reward:+.4f}")
print(f"   RLHF reward model: {REWARD_MODEL}")

print("\n📊 Evaluation Metrics (on held-out test set)")
print(f"   Eval samples     : {NUM_EVAL}")
print(f"   ┌─────────────────────────────────────────────┐")
print(f"   │  Metric           │  Score     │  Ideal      │")
print(f"   ├─────────────────────────────────────────────┤")
print(f"   │  ROUGE-L          │  {avg_rouge_l:.4f}    │  > 0.30     │")
print(f"   │  BERTScore F1     │  {avg_bertscore_f1:.4f}    │  > 0.80     │")
print(f"   │  Perplexity       │  {avg_perplexity:.2f}     │  < 30       │")
print(f"   └─────────────────────────────────────────────┘")

print("\n💾 Saved Artifacts")
print(f"   SFT adapter      : {os.path.abspath(SFT_OUTPUT_DIR)}")
print(f"   RLHF adapter     : {os.path.abspath(PPO_OUTPUT_DIR)}")
print(f"   Backend loads    : {os.path.abspath(FINAL_OUTPUT_DIR)}")

print("\n📋 Sample Predictions")
for i in range(min(3, len(predictions))):
    print(f"\n   Example {i+1}:")
    print(f"   Reference : {references[i][:80]}")
    print(f"   Prediction: {predictions[i][:80]}")

print("\n" + "=" * 65)
print(" 🎉 FinGPT fine-tuning complete! The model is ready for integration.")
print(" 🚀 Restart the MarketLens backend to load the new FinGPT model.")
print("=" * 65)